## © 2026 Noob Dev — Confidential Course Material

This project is part of the **Certified AI Engineer Professional Program** conducted by **Noob Dev**.

---

### ⚠️ Strict Confidentiality & Academic Integrity Notice

All project files — including this notebook, any source code, the knowledge base, the evaluation dataset, and any related materials — are the **exclusive intellectual property of Noob Dev** and are provided **solely for the personal use of enrolled program participants**.

**You may NOT:**

- Share, copy, redistribute, or publish any part of this project in any form
- Upload it to public repositories (GitHub, GitLab, Hugging Face, Kaggle, etc.) or to AI training datasets
- Forward it to non-enrolled individuals, study groups outside the cohort, or online communities
- Use it for commercial purposes outside the program
- Submit it as your own work to any other course, bootcamp, employer, or platform

**Violations constitute academic misconduct and a breach of your program enrollment agreement. Consequences include:**

- Immediate **termination from the Certified AI Engineer Professional Program**
- Forfeiture of all tuition fees, course progress, certifications, and program credentials
- **Legal action** under applicable copyright, trade-secret, and intellectual property laws

By opening, downloading, or using these materials, you acknowledge and accept these terms.

---


# Hints & Common Pitfalls

Read this **before** you start coding, and again whenever you get stuck.

---
## Architecture Decisions to Make Early

Before writing any code, decide:

1. **Embedding model** — OpenAI `text-embedding-3-small` (cheap, fast, paid) vs HuggingFace `all-MiniLM-L6-v2` (free, local, slightly weaker). Pick one and stick with it.
2. **Chunking strategy** — fixed character size with overlap vs LLM-driven semantic chunking. For an intermediate project, **fixed-size with overlap is perfectly fine**. Don't over-engineer.
3. **Vector store path** — pick a folder name (e.g. `vector_db/`) and persist there. Don't rebuild every time you restart the kernel.
4. **LLM** — `gpt-4.1-nano` is cheap and good enough. Avoid `gpt-4` unless you want a big bill. ( or anything as u want )

---
## Common Pitfalls

### Pitfall 1: "My retrieval returns garbage chunks"
- **Check your chunk size.** Too small (< 200 chars) and chunks lose context; too big (> 1500 chars) and embeddings get blurry. Try **500–800 characters with ~100 char overlap** first.
- **Check that you're embedding `page_content`, not the metadata.**
- **Print the top-k chunks for a few test queries** before wiring up the LLM. If retrieval is broken, the LLM cannot save you.

### Pitfall 2: "My citations are hallucinated"
- The LLM will invent sources if you ask it to "cite sources" without giving it the source names in the context.
- **Pass the source filename into the prompt with each chunk.** Something like: `Extract from {source}:\n{content}`.
- Then in the system prompt, instruct the LLM to only cite sources from the provided context.

### Pitfall 3: "Metadata filtering doesn't work"
- Chroma's filter syntax is `where={"category": "products"}`. It must match the exact value you stored in metadata.
- **Print the metadata of a few chunks** to confirm what you actually stored.

### Pitfall 4: "The eval score is 0%"
- If you're using LLM-as-judge, look at the judge's verdicts manually for the first 3 questions. The judge prompt matters a lot.
- If you're using substring match, normalize case and whitespace first.

### Pitfall 5: "Gradio chat forgets history"
- Make sure your chat function accepts `(message, history)` and **passes history into the LLM call**.
- The retrieval should usually be based on the latest message (or a query rewritten using history — stretch goal).

### Pitfall 6: "It works on my machine but the notebook fails on a fresh run"
- Restart the kernel and run all cells top-to-bottom **before** submitting.
- Make sure `.env` is loaded with `load_dotenv(override=True)`.
- Don't rely on variables from a previous notebook.

---
## Quick Sanity Checks

**After ingestion:**
- Print `collection.count()` — does it look reasonable? (Probably 80–180 chunks for this KB.)
- Print one chunk's `metadata` — does it have `source` and `category`?

**After retrieval:**
- For the query "What is the return policy?" — do you get policy chunks back?
- For the query "Tell me about the Nova Phone X1" — do you get product chunks back?
- For a nonsense query like "purple banana spaceship" — what comes back? (Useful for designing your no-context fallback.)

**After generation:**
- Does the answer actually use the retrieved context, or is it generic?
- Do citations point to real files?

---
## Stretch Goal Tips

- **Query rewriting**: Keep the rewrite prompt short. Ask the LLM for "a single focused search query." Don't let it ramble.
- **Reranking**: Retrieve more candidates than you'll use (e.g. retrieve 20, rerank, keep top 5).
- **Hybrid search**: LangChain has `EnsembleRetriever` that combines BM25 + vector retrieval out of the box.
- **Confidence threshold**: Chroma returns distance scores. Pick a threshold by inspecting scores on good vs bad queries.

---
## When to Ask for Help

If you've been stuck on the same bug for **more than 90 minutes**, stop and:

1. Write down (in plain English) exactly what you expected vs what happened.
2. Re-read the relevant Week 5 notebook.
3. Then — and only then — ask on the course channel. Include your written summary.

You'll learn more by struggling for an hour than by being rescued in 5 minutes. But you'll learn nothing by struggling for 5 hours.